# Calibrate the capacity planner against real vLLM on a Colab T4

The planner's absolute numbers rest on three assumed knobs: `mfu`, `mbu` and per-step
overhead. This notebook measures them on a real serving stack:

1. starts `vllm serve Qwen/Qwen2.5-1.5B-Instruct` on the T4 (fp16; the T4 has no bf16),
2. runs `capplan.bench`, a closed-loop client that records prefill TTFT and decode TPOT
   at concurrency 1–64,
3. fits the three knobs with `capplan calibrate` and writes a measured-vs-predicted table,
4. zips `results/` and downloads it.

Runtime: **GPU → T4**. About 20 minutes; it downloads ~3 GB of model weights onto Colab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/kishorhange111/llm-capacity-planner.git
%cd llm-capacity-planner
!pip install -q -e ".[bench]" vllm

Start the server in the background. Prefix caching is off (random prompts would
not hit it anyway) and the prefill chunk is pinned to 2,048 tokens to match
`calibrate --chunk 2048`. If the server fails to start on the T4 with an attention-backend
error, uncomment the `VLLM_ATTENTION_BACKEND` line.

In [ ]:
import os, subprocess, time, urllib.request
# os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
log = open("vllm.log", "w")
srv = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.90", "--max-num-batched-tokens", "2048",
     "--no-enable-prefix-caching", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("server up"); break
    except Exception:
        if srv.poll() is not None:
            raise RuntimeError(open("vllm.log").read()[-3000:])
        time.sleep(5)
else:
    raise RuntimeError("server did not start; see vllm.log")

In [ ]:
!mkdir -p results
!python -m capplan.bench --url http://localhost:8000 --model Qwen/Qwen2.5-1.5B-Instruct \
    --out results/vllm_qwen2.5-1.5b_T4.jsonl

In [ ]:
!capplan calibrate --bench results/vllm_qwen2.5-1.5b_T4.jsonl \
    --model qwen2.5-1.5b --hw T4 --chunk 2048 --out results

In [ ]:
srv.terminate()
!cp vllm.log results/
!nvidia-smi > results/nvidia-smi.txt
!pip freeze | grep -iE "^(vllm|torch|transformers)==" > results/versions.txt
!zip -qr calibration_results.zip results
from google.colab import files
files.download("calibration_results.zip")